# Bangalore DCRNN - Step 4 training (GPU)
Run the cells top to bottom. Needs `bangalore_dcrnn_bundle.zip` (see README).

In [ ]:
import torch, subprocess
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU: Kaggle -> Settings > Accelerator; Colab -> Runtime > Change runtime type"
print(torch.cuda.get_device_name(0))

In [ ]:
import glob, os, shutil, zipfile

ROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
zips = glob.glob("/kaggle/input/**/bangalore_dcrnn_bundle.zip", recursive=True)      + glob.glob("/content/bangalore_dcrnn_bundle.zip")
folders = glob.glob("/kaggle/input/**/bangalore-dcrnn", recursive=True)

if zips:
    zipfile.ZipFile(zips[0]).extractall(ROOT)
elif folders:                       # Kaggle already unzipped it
    shutil.copytree(folders[0], f"{ROOT}/bangalore-dcrnn", dirs_exist_ok=True)
else:
    print("Nothing found. /kaggle/input contains:")
    for p in glob.glob("/kaggle/input/*/*"):
        print("  ", p)
    raise SystemExit("Add the dataset via Input > Add Input, then rerun")

os.chdir(f"{ROOT}/bangalore-dcrnn")
print("project at", os.getcwd())
print(sorted(os.listdir(".")))


In [ ]:
!pip -q install pyyaml python-dotenv scipy

In [ ]:
# 2-epoch smoke test (~1-2 min): must print SMOKE OK before the real run.
!python -m scripts.step4_smoke_test 2>&1 | tail -8

In [ ]:
# Full training. Early stopping (patience 20) + best-val checkpoint guard against overfitting.
!python -u -m scripts.step4_train

In [ ]:
# Learning curves + overfitting check
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv("checkpoints/bangalore_dcrnn/training_log.csv")
ax = log.plot(x="epoch", y=["train_mae","val_mae"], figsize=(8,4), title="MAE (normalized)")
plt.show()
best = log.loc[log.val_mae.idxmin()]
print(best); print("gap val-train:", round(best.val_mae-best.train_mae, 4))
print("val MAE rose after best epoch by:", round(log.val_mae.iloc[-1]-best.val_mae, 4))

In [ ]:
# Package results for download
import shutil
shutil.make_archive("bangalore_dcrnn_results", "zip", "checkpoints/bangalore_dcrnn")
try:
    from google.colab import files; files.download("bangalore_dcrnn_results.zip")
except ImportError:
    print("Kaggle: the zip is in the Output tab (use 'Save Version' > 'Save & Run All')")